<a href="https://colab.research.google.com/github/kapilk75/SmartWater01/blob/main/demoSparc02_07042026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------
# STEP 1: Load Data
# ---------------------------
url = "https://raw.githubusercontent.com/kapilk75/SmartWater01/main/workshop_250_rows.xlsx"
df = pd.read_excel(url)

# ---------------------------
# STEP 2: Standardize Columns
# ---------------------------
df.columns = df.columns.str.strip().str.replace(" ", "_")

# ---------------------------
# STEP 3: Detect Date Column
# ---------------------------
date_cols = [col for col in df.columns if 'date' in col.lower()]

if len(date_cols) == 0:
    raise Exception("No Date column found")

date_col = date_cols[0]
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

# Drop rows with no valid date
df = df.dropna(subset=[date_col])

# ---------------------------
# STEP 4: Sort & Index
# ---------------------------
df = df.sort_values(by=date_col)
df = df.set_index(date_col)

# ---------------------------
# STEP 5: Remove Empty Columns
# ---------------------------
df = df.dropna(axis=1, how='all')

# ---------------------------
# STEP 6: Identify Columns
# ---------------------------
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# ---------------------------
# STEP 7: Missing Value Analysis
# ---------------------------
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)

print("\nMissing % Before:\n", missing_percent.head(10))

# ---------------------------
# STEP 8: Drop Highly Missing Columns (>50%)
# ---------------------------
threshold = 0.5 * len(df)
df = df.dropna(axis=1, thresh=len(df) - threshold)

# Update numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# ---------------------------
# STEP 9: Time-Series Imputation
# ---------------------------

# 1. Time interpolation (BEST)
df[numeric_cols] = df[numeric_cols].interpolate(method='time')

# 2. Rolling smoothing (backup)
df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].rolling(window=3, min_periods=1).mean()
)

# 3. Final fallback
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print("\nMissing After:", df.isnull().sum().sum())

# ---------------------------
# STEP 10: Resample (Daily)
# ---------------------------
df_resampled = df.resample('D').mean(numeric_only=True)

# ---------------------------
# STEP 11: Select Important Features
# ---------------------------
important_features = [col for col in ['TDS', 'pH', 'Turbidity', 'DO'] if col in df_resampled.columns]

if len(important_features) == 0:
    important_features = numeric_cols[:5]  # fallback

print("\nSelected Features:", important_features)


Missing % Before:
 26_If_others_(specify)                                         99.2
27_Time_(Autoset;_Just_touch_next)_(Mostly_empty)              99.2
33_If_others_specify(Mostly)                                   98.8
21_If_others_(specify)                                         98.0
58_Do_you_wash_the_cloth_that_you_use_for_straining?(Empty)    98.0
81_If_others_specify(Empty)                                    96.4
59_How_often_do_you_replace/_wash_the_cloth?(Empty)\n_         96.0
78_If_others_specify(Empty)                                    95.2
48_Is_the_well_covered?(Mostly_empty)                          94.4
47__Location_of_well                                           94.4
dtype: float64

Missing After: 3175

Selected Features: ['lat_2_Location_Click_Upd', 'long_2_Location_Click_Upd', 'accuracy_2_Location_Click_Upd', 'UTM_Northing_2_Location_Click_Upd', 'UTM_Easting_2_Location_Click_Upd']


/tmp/ipykernel_21115/1113204580.py:25: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')


In [3]:
# =========================================================
# FINAL PIPELINE: TIME-SERIES + VISUAL + MAP
# =========================================================

!pip -q install openpyxl plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# ---------------------------
# Load dataset (uploaded file)
# ---------------------------
df = pd.read_excel("/workshop_250_rows.xlsx")

print("Dataset Shape:", df.shape)
print(df.head())

# ---------------------------
# STEP 1: CREATE TIME-SERIES
# ---------------------------

# If no date column → create synthetic timeline
if not any("date" in col.lower() for col in df.columns):
    df["Date"] = pd.date_range(start="2024-01-01", periods=len(df), freq="D")
else:
    for col in df.columns:
        if "date" in col.lower():
            df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")
            df.rename(columns={col: "Date"}, inplace=True)

df = df.sort_values("Date")

# ---------------------------
# STEP 2: CLEAN + IMPUTE
# ---------------------------

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after cleaning:", df.isnull().sum().sum())

# ---------------------------
# STEP 3: CREATE WQI
# ---------------------------
features = [c for c in ["pH", "TDS", "Turbidity", "DO"] if c in df.columns]

if len(features) >= 2:
    norm = (df[features] - df[features].min()) / (df[features].max() - df[features].min())
    df["WQI"] = norm.mean(axis=1)

# ---------------------------
# STEP 4: TIME SERIES VISUAL
# ---------------------------
plt.figure(figsize=(12,6))
plt.plot(df["Date"], df[numeric_cols[0]])
plt.title("Time-Series Trend")
plt.xlabel("Date")
plt.ylabel(numeric_cols[0])
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

# ---------------------------
# STEP 5: KDE DISTRIBUTION
# ---------------------------
plt.figure(figsize=(10,5))
for col in numeric_cols[:4]:
    df[col].plot(kind="kde", label=col)
plt.legend()
plt.title("Density Distribution (KDE)")
plt.grid(alpha=0.3)
plt.show()

# ---------------------------
# STEP 6: SCATTER RELATION
# ---------------------------
if "TDS" in df.columns and "Conductivity" in df.columns:
    plt.figure(figsize=(8,5))
    plt.scatter(df["TDS"], df["Conductivity"], alpha=0.7)
    plt.xlabel("TDS")
    plt.ylabel("Conductivity")
    plt.title("TDS vs Conductivity")
    plt.grid(alpha=0.3)
    plt.show()

# =========================================================
# STEP 7: CREATE GEO DATA (ENGINEERED)
# =========================================================

# If no location column → create synthetic groups
if "Source" not in df.columns:
    df["Source"] = np.random.choice(
        ["Hostel", "Mess", "Restaurant", "Packaged"],
        size=len(df)
    )

# Assign coordinates (Chennai clusters)
location_map = {
    "Hostel": (13.0827, 80.2707),
    "Mess": (13.0845, 80.2715),
    "Restaurant": (13.0800, 80.2690),
    "Packaged": (13.0855, 80.2730)
}

df["latitude"] = df["Source"].map(lambda x: location_map[x][0])
df["longitude"] = df["Source"].map(lambda x: location_map[x][1])

# ---------------------------
# STEP 8: HIGH-CONTRAST MAP
# ---------------------------
fig = px.scatter_mapbox(
    df,
    lat="latitude",
    lon="longitude",
    color="WQI" if "WQI" in df.columns else numeric_cols[0],
    size_max=15,
    zoom=12,
    hover_name="Source",
    hover_data=numeric_cols[:4],
    title="Water Quality Spatial Visualization"
)

fig.update_layout(
    mapbox_style="carto-darkmatter",  # HIGH CONTRAST
    width=1000,
    height=700
)

fig.show()

# ---------------------------
# STEP 9: INTERACTIVE TIME SERIES (PLOTLY)
# ---------------------------
fig2 = px.line(
    df,
    x="Date",
    y=numeric_cols[0],
    title="Interactive Time-Series"
)

fig2.update_layout(template="plotly_dark")
fig2.show()

FileNotFoundError: [Errno 2] No such file or directory: '/workshop_250_rows.xlsx'